# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
- [https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Metadata is an object; access fields as attributes
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

- Each record set defines a table of related data; fields map to specific columns or features.
- In Croissant, entities are referenced by their `@id`.

Let's list all available record sets and their fields (with `@id`).

In [ ]:
# List all record sets with their @id and associated fields
record_sets = dataset.record_sets
if not record_sets:
    print("No record sets found in the metadata.")
else:
    for rs in record_sets:
        print(f"RecordSet: {rs['@id']} ({rs.get('name', 'Unnamed')})")
        if 'field' in rs:
            fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
            for fld in fields:
                print(f"  Field: {fld['@id']} ({fld.get('name','')})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

- Use the record set and field `@id` values found above.
- For demonstration, we extract data from the main survey record set.

For this dataset, we will select the primary record set, which likely contains survey responses. If there are multiple, you can adapt the code as needed.

In [ ]:
# Get list of record set ids
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded {len(df)} records for {record_set_id}")
        else:
            print(f"No records available for record set {record_set_id}.")
    except Exception as e:
        print(f"Error loading record set {record_set_id}: {e}")

# Print available columns in the first DataFrame
if dataframes:
    first_rs_id = list(dataframes.keys())[0]
    print(f"Columns for {first_rs_id}: {dataframes[first_rs_id].columns.tolist()}")
    dataframes[first_rs_id].head()
else:
    print("No dataframes loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

- We select a numeric field (such as `psq_score` or `phq9_score` if present).
- Filter: E.g., show respondents with `psq_score` above a threshold.
- Normalize the selected field.
- Group by an attribute, such as `village` (if present).

All fields are referenced by their `@id`.

In [ ]:
# Exploratory Data Analysis
import numpy as np

# For demonstration, pick the first loaded DataFrame and list numeric columns
if dataframes:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id].copy()

    # Try to automatically detect numeric fields
    numeric_fields = [col for col in df.columns if np.issubdtype(df[col].dtype, np.number)]
    print(f"Numeric fields in {record_set_id}: {numeric_fields}")

    # Use first numeric field for filtering; adjust as needed
    if numeric_fields:
        numeric_field_id = numeric_fields[0]
        threshold = 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Pick a group field, e.g. 'village', 'gender', or any categorical column
        group_fields = [col for col in df.columns if df[col].dtype == 'object']
        if group_fields:
            group_field = group_fields[0]
            if group_field in filtered_df.columns:
                grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
                print(f"Grouped data by {group_field} (mean {numeric_field_id}):")
                print(grouped_df.head())
    else:
        print("No numeric fields found for analysis.")
else:
    print("No data for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

- We'll plot the distribution of a selected numeric field, and visualize group means by a categorical attribute (if available).


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualization section
if dataframes:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]

    # Numeric and group fields, as previously determined
    numeric_fields = [col for col in df.columns if np.issubdtype(df[col].dtype, np.number)]
    group_fields = [col for col in df.columns if df[col].dtype == 'object']

    if numeric_fields:
        numeric_field_id = numeric_fields[0]
        plt.figure(figsize=(8,4))
        sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
        plt.title(f"Distribution of {numeric_field_id}")
        plt.xlabel(numeric_field_id)
        plt.ylabel("Frequency")
        plt.show()

        # Grouped mean plot
        if group_fields:
            group_field = group_fields[0]
            group_means = df.groupby(group_field)[numeric_field_id].mean().reset_index()
            plt.figure(figsize=(10,5))
            sns.barplot(data=group_means, x=group_field, y=numeric_field_id)
            plt.xticks(rotation=45)
            plt.title(f"Mean {numeric_field_id} per {group_field}")
            plt.ylabel(f"Mean {numeric_field_id}")
            plt.show()
    else:
        print("No suitable numeric fields to visualize.")
else:
    print("No data to visualize.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

This notebook demonstrated loading, overview, and processing of survey data on mental health indicators among residents of Kilifi County, Kenya. Using `mlcroissant`, we extracted the dataset, inspected field IDs and types, filtered and normalized scores, grouped by demographic features, and visualized distributions. The Croissant schema and `@id` referencing enabled precise, reproducible data handling.

**Key observations:**
- Data fields include demographics and mental health scores (like GAD-7, PHQ-9, PSQ)
- Visualization and statistical analysis can highlight patterns and potential biases
- Referencing by `@id` ensures consistency and structured exploration

Continue by adapting field selections and EDA as needed for deeper research, modeling, or reporting.